In [ ]:

from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import FrameitRunner
import logging

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import math

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_batsirai.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_belna.yml"))

conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_chido.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_ianos.yml"))

conf.printID()

runner = FrameitRunner(conf)
runner.run()

### By hand

In [ ]:
dico_user = runner.dict_crop_user


### velocities keep their names

In [ ]:
from __future__ import annotations

import numpy as np
import xarray as xr


# ----------------------------
# Core numerics (shift mean)
# ----------------------------
def _pairmean(a: xr.DataArray, b: xr.DataArray, policy: str) -> xr.DataArray:
    if policy == "strict":
        ok = np.isfinite(a) & np.isfinite(b)
        return xr.where(ok, 0.5 * (a + b), np.nan)

    # partial
    wa = xr.where(np.isfinite(a), 1.0, 0.0)
    wb = xr.where(np.isfinite(b), 1.0, 0.0)
    num = xr.where(wa > 0, a, 0.0) + xr.where(wb > 0, b, 0.0)
    den = wa + wb
    return xr.where(den > 0, num / den, np.nan)


def _collocate_keepN(da: xr.DataArray, dim: str, use_forward, policy: str) -> xr.DataArray:
    """
    Keep the same length along `dim`.
      forward  : mean(i, i+1)  -> shift(-1)
      backward : mean(i-1, i)  -> shift(+1)
    Edge handling comes from policy (partial keeps one value, strict gives NaN).
    """
    fwd = _pairmean(da, da.shift({dim: -1}), policy)
    bwd = _pairmean(da.shift({dim: 1}), da, policy)
    return xr.where(use_forward, fwd, bwd)


def _dir_from_coords(c_center: xr.DataArray, c_stag: xr.DataArray, dim_box: str) -> xr.DataArray:
    """
    Return bool DataArray (typically on 'time') indicating forward pairing.
    Forward if median(center - stag) > 0 along the local box axis.
    """
    d = (c_center - c_stag)
    if dim_box in d.dims:
        d = d.median(dim_box, skipna=True)
    return d > 0


def _get_uv_box_dims(da: xr.DataArray) -> tuple[str, str]:
    if da.ndim < 2:
        raise ValueError("Expected at least 2D field for horizontal collocation.")
    return da.dims[-2], da.dims[-1]  # (y_box, x_box) in your extraction


# ----------------------------
# W vertical collocation
# ----------------------------
def _collocate_w_to_zC(
    w: xr.DataArray,
    *,
    zC_name: str,
    zC_coord: xr.DataArray,
    policy: str,
) -> xr.DataArray:
    """
    Return W collocated on zC_name with length = len(zC_coord).

    Cases:
      1) z_w == zC_name           -> no-op (only ensure coords)
      2) len(z_w) == len(zC)      -> keep-N shift mean (direction from coord offsets if possible)
      3) len(z_w) == len(zC)+1    -> true interfaces -> centers (pairmean then drop last)
      4) otherwise                -> collocate on z_w (keep-N) then interpolate to zC_coord
    """
    # If not enough dims to have a vertical dimension, return as-is
    if w.ndim < 3:
        return w

    z_w = w.dims[-3]
    if z_w == zC_name:
        return w.assign_coords({zC_name: zC_coord})

    zW_coord = w.coords.get(z_w, None)
    nzC = zC_coord.sizes[zC_name]
    nzW = w.sizes[z_w]

    if (zW_coord is not None) and (zW_coord.ndim == 1) and (zW_coord.sizes[z_w] == nzW):
        nmin = min(nzC, nzW)
        use_fwd_z = float(np.nanmedian(zC_coord.values[:nmin] - zW_coord.values[:nmin])) > 0
    else:
        use_fwd_z = True

    if nzW == nzC:
        wC = _collocate_keepN(w, z_w, use_fwd_z, policy)
        wC = wC.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return wC

    if nzW == nzC + 1:
        wC = _pairmean(w, w.shift({z_w: -1}), policy).isel({z_w: slice(0, -1)})
        wC = wC.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return wC

    # Fallback: collocate on z_w then interpolate to zC_coord
    w_tmp = _collocate_keepN(w, z_w, use_fwd_z, policy)
    if zW_coord is not None:
        w_tmp = w_tmp.assign_coords({z_w: zW_coord})
        w_i = w_tmp.interp({z_w: zC_coord}, kwargs={"fill_value": np.nan})
        w_i = w_i.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return w_i

    raise ValueError(f"Incompatible vertical sizes and no coord to interpolate: {z_w}={nzW}, {zC_name}={nzC}.")


# ----------------------------
# Main entry point for dico_user
# ----------------------------
def collocate_winds(
    dico_user: dict[str, xr.Dataset],
    *,
    conf,
    policy: str = "partial",          # "partial" | "strict"
    drop_level_w_group: bool = True,  # MNH only, to avoid confusion
) -> dict[str, xr.Dataset]:
    """
    Post-extraction collocation for MNH (and no-op for AROME).

    Behavior
    --------
    - The dictionary key for the main 3D atmospheric group is assumed to be conf.name_vertical_dim
      (e.g., "level"). Only this dataset is replaced in the returned dict.
    - The wind variables keep their original names (e.g., UT, VT, WT). They are overwritten
      by their collocated versions in-place within the returned dataset.
    - W may come from the main group or from a separate group (e.g., "level_w"). In the latter case,
      it is collocated vertically onto the conf.name_vertical_dim grid and inserted into the main group.
    - Optionally drops the separate level_w group (MNH), because WT is injected into the main group.

    Minimal assumptions
    -------------------
    - horizontal dims of winds are the last two dims (typically y_box, x_box after extraction)
    - MNH: dataset contains coords {ni, ni_u} on (time, x_box) and {nj, nj_v} on (time, y_box)
    """
    if policy not in {"partial", "strict"}:
        raise ValueError("policy must be 'partial' or 'strict'.")

    atm_model = str(getattr(conf, "atm_model")).upper()
    zC_name = getattr(conf, "name_vertical_dim")  # also used as dict key (e.g., "level")

    if zC_name not in dico_user:
        return dico_user

    ds_h = dico_user[zC_name].copy(deep=False)
    out = ds_h.copy(deep=False)

    # Wind variable names from conf (e.g., UT/VT/WT for MNH, u/v/w for other models)
    vel = getattr(conf, "velocity_aliases", {}) or {}
    u_name = vel.get("u_velocity", "u")
    v_name = vel.get("v_velocity", "v")
    w_name = vel.get("w_velocity", "w")

    # Identify where W lives (same group or separate group)
    ds_w = None
    key_w = None
    if w_name in out.data_vars:
        ds_w = out
        key_w = zC_name
    else:
        for k, ds in dico_user.items():
            if (w_name in ds.data_vars) and ("level_w" in ds.coords):
                ds_w = ds
                key_w = k
                break

    # Vertical coordinate must exist in main group for WT insertion
    if zC_name not in out.coords:
        raise ValueError(f"Main 3D group is missing vertical coordinate {zC_name!r}.")

    # ----------------------------
    # U, V (overwrite with collocated versions, keep original names)
    # ----------------------------
    if (u_name in out.data_vars) and (v_name in out.data_vars):
        u = out[u_name]
        v = out[v_name]
        y_dim_u, x_dim_u = _get_uv_box_dims(u)
        y_dim_v, x_dim_v = _get_uv_box_dims(v)

        if atm_model == "MNH":
            # Use ni/ni_u and nj/nj_v if present, otherwise default forward
            xC = getattr(conf, "name_lon_dim", "ni")
            yC = getattr(conf, "name_lat_dim", "nj")
            xU = f"{xC}_u"
            yV = f"{yC}_v"

            use_fwd_x = True
            use_fwd_y = True
            if (xC in out.coords) and (xU in out.coords) and (x_dim_u in out.coords[xC].dims):
                use_fwd_x = _dir_from_coords(out.coords[xC], out.coords[xU], x_dim_u)
            if (yC in out.coords) and (yV in out.coords) and (y_dim_v in out.coords[yC].dims):
                use_fwd_y = _dir_from_coords(out.coords[yC], out.coords[yV], y_dim_v)

            out[u_name] = _collocate_keepN(u, x_dim_u, use_fwd_x, policy).rename(u_name)
            out[v_name] = _collocate_keepN(v, y_dim_v, use_fwd_y, policy).rename(v_name)
        else:
            # AROME (or others): assumed already on mass point in your workflow (no-op)
            out[u_name] = u.rename(u_name)
            out[v_name] = v.rename(v_name)

    # ----------------------------
    # W (overwrite with collocated version, keep original name)
    # ----------------------------
    if (ds_w is not None) and (w_name in ds_w.data_vars):
        w = ds_w[w_name]
        wC = _collocate_w_to_zC(
            w,
            zC_name=zC_name,
            zC_coord=out.coords[zC_name],
            policy=policy,
        )
        out[w_name] = wC.rename(w_name)

        # Ensure WT shares the same horizontal coords as the main group, when possible
        for cname in ("x_box", "y_box", "x_box_km", "y_box_km", "latitude", "longitude"):
            if cname in out.coords and cname not in out[w_name].coords:
                out[w_name] = out[w_name].assign_coords({cname: out.coords[cname]})

    # Build output dict with only the main group replaced
    dico_out = dict(dico_user)
    dico_out[zC_name] = out

    # Optionally drop the separate level_w group (MNH only)
    if (
        drop_level_w_group
        and atm_model == "MNH"
        and key_w is not None
        and key_w != zC_name
    ):
        dico_out.pop(key_w, None)

    return dico_out


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Core operators (same logic as collocation step)
# ---------------------------------------------------------------------
def _pairmean(a: xr.DataArray, b: xr.DataArray, policy: str) -> xr.DataArray:
    if policy == "strict":
        ok = np.isfinite(a) & np.isfinite(b)
        return xr.where(ok, 0.5 * (a + b), np.nan)

    wa = xr.where(np.isfinite(a), 1.0, 0.0)
    wb = xr.where(np.isfinite(b), 1.0, 0.0)
    num = xr.where(wa > 0, a, 0.0) + xr.where(wb > 0, b, 0.0)
    den = wa + wb
    return xr.where(den > 0, num / den, np.nan)


def _collocate_keepN(da: xr.DataArray, dim: str, use_forward: xr.DataArray, policy: str) -> xr.DataArray:
    fwd = _pairmean(da, da.shift({dim: -1}), policy)        # (i, i+1)
    bwd = _pairmean(da.shift({dim: 1}), da, policy)         # (i-1, i)
    return xr.where(use_forward, fwd, bwd)


def _use_forward_from_coords(c_center: xr.DataArray, c_stag: xr.DataArray, dim_box: str) -> xr.DataArray:
    d = (c_center - c_stag)
    if dim_box in d.dims:
        d = d.median(dim_box, skipna=True)
    return d > 0


def _find_dataset_with_var(dico: dict[str, xr.Dataset], varname: str) -> tuple[str, xr.Dataset] | tuple[None, None]:
    for k, ds in dico.items():
        if varname in ds.data_vars:
            return k, ds
    return None, None


def _collocate_w_to_zC_ref(
    w: xr.DataArray,
    *,
    zC_name: str,
    zC_coord: xr.DataArray,
    policy: str,
) -> xr.DataArray:
    """
    Reference WT on zC_name with len(zC_coord), same cases as collocation.
    """
    if w.ndim < 3:
        return w

    z_w = w.dims[-3]
    if z_w == zC_name:
        return w.assign_coords({zC_name: zC_coord})

    zW_coord = w.coords.get(z_w, None)
    nzC = zC_coord.sizes[zC_name]
    nzW = w.sizes[z_w]

    if (zW_coord is not None) and (zW_coord.ndim == 1) and (zW_coord.sizes[z_w] == nzW):
        nmin = min(nzC, nzW)
        use_fwd_z = float(np.nanmedian(zC_coord.values[:nmin] - zW_coord.values[:nmin])) > 0
    else:
        use_fwd_z = True

    if nzW == nzC:
        wC = _collocate_keepN(w, z_w, xr.DataArray(use_fwd_z), policy)
        return wC.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})

    if nzW == nzC + 1:
        wC = _pairmean(w, w.shift({z_w: -1}), policy).isel({z_w: slice(0, -1)})
        return wC.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})

    # Fallback: collocate on z_w then interpolate to zC
    w_tmp = _collocate_keepN(w, z_w, xr.DataArray(use_fwd_z), policy)
    if zW_coord is not None:
        w_tmp = w_tmp.assign_coords({z_w: zW_coord})
        w_i = w_tmp.interp({z_w: zC_coord}, kwargs={"fill_value": np.nan})
        return w_i.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})

    raise ValueError(f"Incompatible vertical sizes and no coord to interpolate: {z_w}={nzW}, {zC_name}={nzC}.")


# ---------------------------------------------------------------------
# Quick verification plots + metrics
# ---------------------------------------------------------------------
def quickcheck_collocation(
    dico_collocated: dict[str, xr.Dataset],
    dico_raw: dict[str, xr.Dataset],
    *,
    conf,
    policy: str = "partial",
    t: int = 0,
    k: int = 0,
    y_line: int | None = None,
    x_line: int | None = None,
):
    """
    Minimal checks when collocation overwrites winds without renaming.

    Checks:
      1) Rebuild expected UT, VT from raw UT, VT and compare to collocated UT, VT.
      2) 2D maps: raw, collocated, (collocated - reference) at one time, one level if 3D.
      3) 1D transect: raw UT, neighbor-shifted UT, collocated UT.
      4) If WT exists: rebuild expected WT on zC and compare to collocated WT, with one vertical profile.
    """
    zkey = conf.name_vertical_dim
    if zkey not in dico_collocated or zkey not in dico_raw:
        raise KeyError(f"Missing vertical group {zkey!r} in collocated or raw dict.")

    dsC = dico_collocated[zkey]
    dsR = dico_raw[zkey]

    vel = getattr(conf, "velocity_aliases", {}) or {}
    u_name = vel.get("u_velocity") or "UT"
    v_name = vel.get("v_velocity") or "VT"
    w_name = vel.get("w_velocity") or "WT"

    if (u_name not in dsR.data_vars) or (v_name not in dsR.data_vars):
        raise KeyError(f"Raw dataset is missing {u_name!r} and/or {v_name!r}.")

    if (u_name not in dsC.data_vars) or (v_name not in dsC.data_vars):
        raise KeyError(f"Collocated dataset is missing {u_name!r} and/or {v_name!r}.")

    UT_raw = dsR[u_name]
    VT_raw = dsR[v_name]
    UT_col = dsC[u_name]
    VT_col = dsC[v_name]

    # Identify dims
    ydim, xdim = UT_raw.dims[-2], UT_raw.dims[-1]
    zdim = UT_raw.dims[-3] if UT_raw.ndim >= 3 else None

    if y_line is None:
        y_line = UT_raw.sizes[ydim] // 2
    if x_line is None:
        x_line = UT_raw.sizes[xdim] // 2

    # Direction from coords (MNH case)
    xC = getattr(conf, "name_lon_dim", "ni")
    yC = getattr(conf, "name_lat_dim", "nj")
    xU = f"{xC}_u"
    yV = f"{yC}_v"

    for cname in (xC, xU, yC, yV):
        if cname not in dsR.coords:
            raise ValueError(f"Missing coord {cname!r} in raw dataset for direction inference.")

    use_fwd_x = _use_forward_from_coords(dsR[xC], dsR[xU], xdim)
    use_fwd_y = _use_forward_from_coords(dsR[yC], dsR[yV], ydim)

    # Reference collocation from raw
    UT_ref = _collocate_keepN(UT_raw, xdim, use_fwd_x, policy)
    VT_ref = _collocate_keepN(VT_raw, ydim, use_fwd_y, policy)

    # Slice helper
    def _slice_2d(da: xr.DataArray) -> xr.DataArray:
        if "time" in da.dims:
            da = da.isel(time=t)
        if (zdim is not None) and (zdim in da.dims):
            da = da.isel({zdim: k})
        return da

    UT2_raw = _slice_2d(UT_raw)
    UT2_col = _slice_2d(UT_col)
    UT2_ref = _slice_2d(UT_ref)
    UT2_err = UT2_col - UT2_ref

    VT2_raw = _slice_2d(VT_raw)
    VT2_col = _slice_2d(VT_col)
    VT2_ref = _slice_2d(VT_ref)
    VT2_err = VT2_col - VT2_ref

    # Metrics
    def _metrics(err2: xr.DataArray, label: str) -> None:
        v = np.asarray(err2.values)
        print(label)
        print("  max|err|:", float(np.nanmax(np.abs(v))))
        print("  mean|err|:", float(np.nanmean(np.abs(v))))

    print(f"Collocation check at t={t}, k={k} (if applicable)")
    _metrics(UT2_err, f"{u_name}: collocated - reference")
    _metrics(VT2_err, f"{v_name}: collocated - reference")

    # 2D plots: raw, collocated, error
    def _plot3(a, b, c, title_prefix: str) -> None:
        plt.figure()
        plt.imshow(a.values)
        plt.title(f"{title_prefix} raw")
        plt.colorbar()
        plt.show()

        plt.figure()
        plt.imshow(b.values)
        plt.title(f"{title_prefix} collocated")
        plt.colorbar()
        plt.show()

        plt.figure()
        plt.imshow(c.values)
        plt.title(f"{title_prefix} collocated - reference")
        plt.colorbar()
        plt.show()

    _plot3(UT2_raw, UT2_col, UT2_err, u_name)
    _plot3(VT2_raw, VT2_col, VT2_err, v_name)

    # 1D transect along x at fixed y
    UT_line_raw = UT2_raw.isel({ydim: y_line})
    UT_line_col = UT2_col.isel({ydim: y_line})

    if bool(use_fwd_x.isel(time=t).values) if "time" in use_fwd_x.dims else bool(use_fwd_x.values):
        UT_nei = UT_line_raw.shift({xdim: -1})
        lab_nei = f"{u_name} raw shifted -1"
    else:
        UT_nei = UT_line_raw.shift({xdim: 1})
        lab_nei = f"{u_name} raw shifted +1"

    plt.figure()
    plt.plot(UT_line_raw.values, label=f"{u_name} raw")
    plt.plot(UT_nei.values, label=lab_nei)
    plt.plot(UT_line_col.values, label=f"{u_name} collocated")
    plt.title(f"X-transect at y={y_line}, t={t}" + (f", k={k}" if zdim is not None else ""))
    plt.legend()
    plt.show()

    # -----------------------------------------------------------------
    # Vertical check for WT, if available
    # -----------------------------------------------------------------
    if w_name in dsC.data_vars:
        WT_col = dsC[w_name]
        # WT raw can be in main group or in a separate group
        _, dsW = _find_dataset_with_var(dico_raw, w_name)
        if dsW is None:
            print(f"WT check skipped: {w_name!r} not found in raw dict.")
            return

        WT_raw = dsW[w_name]
        zC_coord = dsC.coords[zkey]
        WT_ref = _collocate_w_to_zC_ref(WT_raw, zC_name=zkey, zC_coord=zC_coord, policy=policy)

        WT_prof_raw = WT_raw.isel(time=t, **{ydim: y_line, xdim: x_line}) if "time" in WT_raw.dims else WT_raw.isel(**{ydim: y_line, xdim: x_line})
        WT_prof_col = WT_col.isel(time=t, **{ydim: y_line, xdim: x_line}) if "time" in WT_col.dims else WT_col.isel(**{ydim: y_line, xdim: x_line})
        WT_prof_ref = WT_ref.isel(time=t, **{ydim: y_line, xdim: x_line}) if "time" in WT_ref.dims else WT_ref.isel(**{ydim: y_line, xdim: x_line})

        # Profile plot (z axis depends on the dim used)
        z_prof_dim_raw = WT_prof_raw.dims[0] if WT_prof_raw.ndim == 1 else WT_prof_raw.dims[-1]
        z_prof_dim_col = zkey

        plt.figure()
        if (z_prof_dim_raw in WT_prof_raw.coords) and (WT_prof_raw.coords[z_prof_dim_raw].ndim == 1):
            plt.plot(WT_prof_raw.values, WT_prof_raw.coords[z_prof_dim_raw].values, label=f"{w_name} raw")
        else:
            plt.plot(WT_prof_raw.values, np.arange(WT_prof_raw.size), label=f"{w_name} raw")

        plt.plot(WT_prof_col.values, WT_prof_col.coords[z_prof_dim_col].values, label=f"{w_name} collocated")
        plt.plot(WT_prof_ref.values, WT_prof_ref.coords[z_prof_dim_col].values, label=f"{w_name} reference")

        plt.title(f"WT vertical profile at (x={x_line}, y={y_line}), t={t}")
        plt.xlabel("m s-1")
        plt.ylabel("vertical coordinate")
        plt.legend()
        plt.show()

        WT_err2 = _slice_2d(WT_col) - _slice_2d(WT_ref)
        _metrics(WT_err2, f"{w_name}: collocated - reference")
    else:
        print(f"WT check skipped: {w_name!r} not present in collocated dataset.")


# ---------------------------------------------------------------------
# Typical usage
# ---------------------------------------------------------------------
# dico_collocated = collocate_winds(dico_user, conf=conf, policy="partial")
# quickcheck_collocation(dico_collocated, dico_user, conf=conf, policy="partial", t=0, k=0)

In [ ]:
dico_collocated = collocate_winds(dico_user, conf=conf, policy="partial")

In [ ]:
quickcheck_collocation(dico_collocated, dico_user, conf=conf, policy="partial", t=0, k=0)

### Velocity are named uc,vc,wc

In [ ]:
import numpy as np
import xarray as xr


# ----------------------------
# Core numerics (shift mean)
# ----------------------------
def _pairmean(a: xr.DataArray, b: xr.DataArray, policy: str) -> xr.DataArray:
    if policy == "strict":
        ok = np.isfinite(a) & np.isfinite(b)
        return xr.where(ok, 0.5 * (a + b), np.nan)

    # partial
    wa = xr.where(np.isfinite(a), 1.0, 0.0)
    wb = xr.where(np.isfinite(b), 1.0, 0.0)
    num = xr.where(wa > 0, a, 0.0) + xr.where(wb > 0, b, 0.0)
    den = wa + wb
    return xr.where(den > 0, num / den, np.nan)


def _collocate_keepN(da: xr.DataArray, dim: str, use_forward, policy: str) -> xr.DataArray:
    """
    Keep the same length along `dim`.
      forward  : mean(i, i+1)  -> shift(-1)
      backward : mean(i-1, i)  -> shift(+1)
    Edge handling comes from policy (partial keeps one value, strict gives NaN).
    """
    fwd = _pairmean(da, da.shift({dim: -1}), policy)
    bwd = _pairmean(da.shift({dim: 1}), da, policy)
    return xr.where(use_forward, fwd, bwd)


def _dir_from_coords(c_center: xr.DataArray, c_stag: xr.DataArray, dim_box: str) -> xr.DataArray:
    """
    Return bool DataArray (typically on 'time') indicating forward pairing.
    Forward if median(center - stag) > 0 along the local box axis.
    """
    d = (c_center - c_stag)
    if dim_box in d.dims:
        d = d.median(dim_box, skipna=True)
    return d > 0


def _get_uv_box_dims(da: xr.DataArray) -> tuple[str, str]:
    if da.ndim < 2:
        raise ValueError("Expected at least 2D field for horizontal collocation.")
    return da.dims[-2], da.dims[-1]  # (y_box, x_box) in your extraction


# ----------------------------
# W vertical collocation
# ----------------------------
def _collocate_w_to_zC(
    w: xr.DataArray,
    *,
    zC_name: str,
    zC_coord: xr.DataArray,
    policy: str,
) -> xr.DataArray:
    """
    Produce wc on zC_name with length = len(zC_coord), robust to common level_w/level patterns.

    Cases:
      1) z_w == zC_name           -> wc = w (no-op)
      2) len(z_w) == len(zC)      -> keep-N shift mean (direction from coord offsets if possible)
      3) len(z_w) == len(zC)+1    -> true interfaces -> centers (pairmean then drop last)
      4) otherwise                -> collocate on z_w (keep-N) then interpolate to zC_coord
    """
    if w.ndim < 3:
        return w.rename("wc")

    z_w = w.dims[-3]
    if z_w == zC_name:
        wc = w
        wc = wc.assign_coords({zC_name: zC_coord})
        return wc.rename("wc")

    zW_coord = w.coords.get(z_w, None)
    nzC = zC_coord.sizes[zC_name]
    nzW = w.sizes[z_w]

    if (zW_coord is not None) and (zW_coord.ndim == 1) and (zW_coord.sizes[z_w] == nzW):
        use_fwd_z = (float(np.nanmedian(zC_coord.values[: min(nzC, nzW)] - zW_coord.values[: min(nzC, nzW)])) > 0)
    else:
        use_fwd_z = True

    if nzW == nzC:
        wc = _collocate_keepN(w, z_w, use_fwd_z, policy)
        wc = wc.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return wc.rename("wc")

    if nzW == nzC + 1:
        wc = _pairmean(w, w.shift({z_w: -1}), policy).isel({z_w: slice(0, -1)})
        wc = wc.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return wc.rename("wc")

    # Fallback: collocate on z_w then interpolate to zC_coord
    wc_tmp = _collocate_keepN(w, z_w, use_fwd_z, policy)
    if zW_coord is not None:
        wc_tmp = wc_tmp.assign_coords({z_w: zW_coord})
        wc_i = wc_tmp.interp({z_w: zC_coord}, kwargs={"fill_value": np.nan})
        wc_i = wc_i.rename({z_w: zC_name}).assign_coords({zC_name: zC_coord})
        return wc_i.rename("wc")

    raise ValueError(f"Incompatible vertical sizes and no coord to interpolate: {z_w}={nzW}, {zC_name}={nzC}.")


# ----------------------------
# Main entry point for dico_user
# ----------------------------
def collocate_winds(
    dico_user: dict[str, xr.Dataset],
    *,
    conf,
    policy: str = "partial",   # "partial" | "strict"
    drop_original: bool = True,
    drop_level_w_group: bool = True,
) -> dict[str, xr.Dataset]:
    """
    Concise post-extraction collocation for both MNH and AROME.

    It replaces only the 'height' group (key = conf.name_vertical_dim) with a dataset that includes:
      - uc, vc (for MNH: computed with shift along x_box/y_box; for AROME: aliases of u,v)
      - wc (if w exists, computed robustly on the same vertical grid as the height group)

    Minimal assumptions:
      - horizontal dims of winds are the last two dims (typically y_box, x_box after your extraction)
      - for MNH, extracted dataset contains coords {ni, ni_u} on (time, x_box) and {nj, nj_v} on (time, y_box)
    """
    if policy not in {"partial", "strict"}:
        raise ValueError("policy must be 'partial' or 'strict'.")

    atm_model = str(getattr(conf, "atm_model")).upper()
    zC_name = getattr(conf, "name_vertical_dim")  # also used as dict key in your pipeline

    if zC_name not in dico_user:
        return dico_user

    ds_h = dico_user[zC_name].copy(deep=False)

    vel = getattr(conf, "velocity_aliases", {})
    u_name = vel.get("u_velocity", "u")
    v_name = vel.get("v_velocity", "v")
    w_name = vel.get("w_velocity", "w")
    
    # Locate W dataset and remember its dict key
    ds_w = None
    key_w = None
    if w_name in ds_h.data_vars:
        ds_w = ds_h
        key_w = zC_name
    else:
        for k, ds in dico_user.items():
            if (w_name in ds.data_vars) and ("level_w" in ds.coords):
                ds_w = ds
                key_w = k
                break

    out = ds_h.copy(deep=False)

    # Vertical coordinate of the height group must exist for robust wc insertion
    if zC_name not in out.coords:
        raise ValueError(f"Height group is missing vertical coordinate {zC_name!r}.")

    # ----------------------------
    # uc, vc
    # ----------------------------
    if (u_name in out.data_vars) and (v_name in out.data_vars):
        u = out[u_name]
        v = out[v_name]
        y_dim_u, x_dim_u = _get_uv_box_dims(u)
        y_dim_v, x_dim_v = _get_uv_box_dims(v)

        if atm_model == "MNH":
            # Use ni/ni_u and nj/nj_v if present, otherwise default forward
            xC = getattr(conf, "name_lon_dim", "ni")
            yC = getattr(conf, "name_lat_dim", "nj")
            xU = f"{xC}_u"
            yV = f"{yC}_v"

            use_fwd_x = True
            use_fwd_y = True
            if (xC in out.coords) and (xU in out.coords) and (x_dim_u in out.coords[xC].dims):
                use_fwd_x = _dir_from_coords(out.coords[xC], out.coords[xU], x_dim_u)
            if (yC in out.coords) and (yV in out.coords) and (y_dim_v in out.coords[yC].dims):
                use_fwd_y = _dir_from_coords(out.coords[yC], out.coords[yV], y_dim_v)

            out["uc"] = _collocate_keepN(u, x_dim_u, use_fwd_x, policy).rename("uc")
            out["vc"] = _collocate_keepN(v, y_dim_v, use_fwd_y, policy).rename("vc")
        else:
            # AROME: u,v are already on mass point in your workflow, keep as aliases
            out["uc"] = u.rename("uc")
            out["vc"] = v.rename("vc")

    # ----------------------------
    # wc
    # ----------------------------
    if (ds_w is not None) and (w_name in ds_w.data_vars):
        w = ds_w[w_name]
        out["wc"] = _collocate_w_to_zC(
            w,
            zC_name=zC_name,
            zC_coord=out.coords[zC_name],
            policy=policy,
        )

        # Ensure wc has the same horizontal coords as the height group when possible
        for cname in ("x_box", "y_box", "x_box_km", "y_box_km", "latitude", "longitude"):
            if cname in out.coords and cname not in out["wc"].coords:
                out["wc"] = out["wc"].assign_coords({cname: out.coords[cname]})

    # Optional: drop original winds in the returned dict (keep dico_user intact)
    if drop_original:
        out = out.drop_vars([u_name, v_name, w_name], errors="ignore")

    # Return new dict with only height group replaced
    dico_out = dict(dico_user)
    dico_out[zC_name] = out
    
    # Drop the separate level_w group to avoid confusion (MNH only)
    if (
        drop_level_w_group
        and atm_model == "MNH"
        and key_w is not None
        and key_w != zC_name
    ):
        dico_out.pop(key_w, None)
    
    return dico_out

### quick test

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Core operators (same logic as your collocation step)
# ---------------------------------------------------------------------
def _pairmean(a: xr.DataArray, b: xr.DataArray, policy: str) -> xr.DataArray:
    if policy == "strict":
        ok = np.isfinite(a) & np.isfinite(b)
        return xr.where(ok, 0.5 * (a + b), np.nan)
    wa = xr.where(np.isfinite(a), 1.0, 0.0)
    wb = xr.where(np.isfinite(b), 1.0, 0.0)
    num = xr.where(wa > 0, a, 0.0) + xr.where(wb > 0, b, 0.0)
    den = wa + wb
    return xr.where(den > 0, num / den, np.nan)


def _collocate_keepN(da: xr.DataArray, dim: str, use_forward: xr.DataArray, policy: str) -> xr.DataArray:
    fwd = _pairmean(da, da.shift({dim: -1}), policy)        # (i, i+1)
    bwd = _pairmean(da.shift({dim: 1}), da, policy)         # (i-1, i)
    return xr.where(use_forward, fwd, bwd)


def _use_forward_from_coords(c_center: xr.DataArray, c_stag: xr.DataArray, dim_box: str) -> xr.DataArray:
    d = (c_center - c_stag)
    if dim_box in d.dims:
        d = d.median(dim_box, skipna=True)
    return d > 0


def _find_dataset_with_var(dico: dict[str, xr.Dataset], varname: str) -> xr.Dataset | None:
    for ds in dico.values():
        if varname in ds.data_vars:
            return ds
    return None


# ---------------------------------------------------------------------
# Quick verification plots + metrics
# ---------------------------------------------------------------------
def quickcheck_collocation(
    dico_collocated: dict[str, xr.Dataset],
    dico_raw: dict[str, xr.Dataset],
    *,
    conf,
    policy: str = "partial",
    t: int = 0,
    k: int = 0,
    y_line: int | None = None,
    x_line: int | None = None,
):
    """
    Minimal checks:
      1) Rebuild expected uc/vc from UT/VT and compare to uc/vc.
      2) Plot UT, uc, and (uc - expected) at one time and one level.
      3) Plot a 1D transect showing UT, shifted UT, and uc.
      4) If WT + wc exist, do the same vertically.

    Important: run collocation with drop_original=False to keep UT/VT/WT in outputs, or provide dico_raw.
    """
    zkey = conf.name_vertical_dim
    ds = dico_collocated[zkey]

    vel = conf.velocity_aliases
    u_name = vel.get("u_velocity", "UT")
    v_name = vel.get("v_velocity", "VT")
    w_name = vel.get("w_velocity", "WT")

    # Ensure originals are available (from ds or raw dict)
    ds_raw_h = dico_raw[zkey]
    UT = ds.get(u_name, ds_raw_h.get(u_name))
    VT = ds.get(v_name, ds_raw_h.get(v_name))
    uc = ds.get("uc", None)
    vc = ds.get("vc", None)

    if UT is None or VT is None or uc is None or vc is None:
        raise ValueError(
            "Missing UT/VT or uc/vc. For checks, run collocate_winds(..., drop_original=False) "
            "or make sure dico_raw contains UT/VT and dico_collocated contains uc/vc."
        )

    # Identify box dims
    ydim, xdim = UT.dims[-2], UT.dims[-1]
    if y_line is None:
        y_line = UT.sizes[ydim] // 2
    if x_line is None:
        x_line = UT.sizes[xdim] // 2

    # Direction from coords (MNH case)
    xC = getattr(conf, "name_lon_dim", "ni")
    yC = getattr(conf, "name_lat_dim", "nj")
    xU = f"{xC}_u"
    yV = f"{yC}_v"

    if (xC not in ds.coords) or (xU not in ds.coords) or (yC not in ds.coords) or (yV not in ds.coords):
        raise ValueError(f"Missing coords among {xC},{xU},{yC},{yV} in extracted dataset.")

    use_fwd_x = _use_forward_from_coords(ds[xC], ds[xU], xdim)
    use_fwd_y = _use_forward_from_coords(ds[yC], ds[yV], ydim)

    # Expected collocation from raw UT/VT
    uc_ref = _collocate_keepN(UT, xdim, use_fwd_x, policy).rename("uc_ref")
    vc_ref = _collocate_keepN(VT, ydim, use_fwd_y, policy).rename("vc_ref")

    # Error metrics (lazy-friendly)
    err_uc = (uc - uc_ref).isel(time=t, **{UT.dims[-3]: k})
    err_vc = (vc - vc_ref).isel(time=t, **{VT.dims[-3]: k})

    print("uc check, time index", t, "level index", k)
    print("max|uc-uc_ref|:", float(np.nanmax(np.abs(err_uc.values))))
    print("mean|uc-uc_ref|:", float(np.nanmean(np.abs(err_uc.values))))
    print("vc check, time index", t, "level index", k)
    print("max|vc-vc_ref|:", float(np.nanmax(np.abs(err_vc.values))))
    print("mean|vc-vc_ref|:", float(np.nanmean(np.abs(err_vc.values))))

    # 2D plots: UT, uc, error uc
    UT2 = UT.isel(time=t, **{UT.dims[-3]: k})
    uc2 = uc.isel(time=t, **{uc.dims[-3]: k})
    uc_err2 = (uc2 - uc_ref.isel(time=t, **{uc_ref.dims[-3]: k}))

    plt.figure()
    plt.imshow(UT2.values)
    plt.title(f"{u_name} at t={t}, k={k}")
    plt.colorbar()
    plt.show()

    plt.figure()
    plt.imshow(uc2.values)
    plt.title(f"uc at t={t}, k={k}")
    plt.colorbar()
    plt.show()

    plt.figure()
    plt.imshow(uc_err2.values)
    plt.title(f"uc - uc_ref at t={t}, k={k}")
    plt.colorbar()
    plt.show()

    # 1D transect along x at y_line: UT, shifted UT, uc
    UT_line = UT2.isel({ydim: y_line})
    uc_line = uc2.isel({ydim: y_line})

    if bool(use_fwd_x.isel(time=t).values):
        UT_nei = UT_line.shift({xdim: -1})
        lab_nei = f"{u_name} shifted -1"
    else:
        UT_nei = UT_line.shift({xdim: 1})
        lab_nei = f"{u_name} shifted +1"

    plt.figure()
    plt.plot(UT_line.values, label=u_name)
    plt.plot(UT_nei.values, label=lab_nei)
    plt.plot(uc_line.values, label="uc")
    plt.title(f"X-transect at y={y_line}, t={t}, k={k}")
    plt.legend()
    plt.show()

    # Vertical check for w, if available
    wc = ds.get("wc", None)
    ds_w_raw = _find_dataset_with_var(dico_raw, w_name)
    if wc is not None and ds_w_raw is not None:
        W = ds.get(w_name, ds_w_raw.get(w_name))
        if W is not None:
            zW = W.dims[-3]
            zCdim = wc.dims[-3]  # should be conf.name_vertical_dim

            # Build reference wc on the same vertical dim length as wc
            if W.sizes[zW] == ds[zkey].sizes[zkey]:
                wc_ref = _collocate_keepN(W, zW, xr.DataArray(True), policy).rename({zW: zkey})
                wc_ref = wc_ref.assign_coords({zkey: ds[zkey]})
            elif W.sizes[zW] == ds[zkey].sizes[zkey] + 1:
                wc_ref = _pairmean(W, W.shift({zW: -1}), policy).isel({zW: slice(0, -1)}).rename({zW: zkey})
                wc_ref = wc_ref.assign_coords({zkey: ds[zkey]})
            else:
                wc_ref = None

            if wc_ref is not None:
                W_prof = W.isel(time=t, **{ydim: y_line, xdim: x_line})
                wc_prof = wc.isel(time=t, **{ydim: y_line, xdim: x_line})
                wc_ref_prof = wc_ref.isel(time=t, **{ydim: y_line, xdim: x_line})

                plt.figure()
                plt.plot(W_prof.values, W_prof[zW].values, label=w_name)
                plt.plot(wc_prof.values, wc_prof[zkey].values, label="wc")
                plt.plot(wc_ref_prof.values, wc_ref_prof[zkey].values, label="wc_ref")
                plt.title(f"Vertical profile at (x={x_line}, y={y_line}), t={t}")
                plt.xlabel("m s-1")
                plt.ylabel("z (model units)")
                plt.legend()
                plt.show()


# ---------------------------------------------------------------------
# Typical usage
# ---------------------------------------------------------------------
# 1) For verification, keep originals in collocated output
# dico_collocated = collocate_winds(dico_user, conf=conf, policy="partial", drop_original=False)
#
# 2) Run checks
# quickcheck_collocation(dico_collocated, dico_user, conf=conf, policy="partial", t=0, k=0)

In [ ]:
quickcheck_collocation(dico_collocated, dico_user, conf=conf, policy="partial", t=2, k=4)

### TEST with xgcm

In [ ]:
from __future__ import annotations

from typing import Mapping, MutableMapping, Optional
import logging

import numpy as np
import xarray as xr
from xgcm import Grid


def collocate_winds(
    dico_user: Mapping[str, xr.Dataset],
    *,
    conf,
    boundary_policy: str = "partial",  # "partial" | "strict"
    logger: Optional[logging.Logger] = None,
) -> Mapping[str, xr.Dataset]:
    """
    Collocate winds to cell centers in the height-coordinate group only.

    Model policy
    ------------
    - MNH: always use xgcm to collocate u, v, w to (nj, ni, level) centers.
    - AROME: always fast-path, only collocate w vertically if needed.

    Input
    -----
    dico_user: dict-like {group_key: xr.Dataset}
        group_key is expected to match conf.name_vertical_dim for the height group
        (e.g. "heightAboveGround" for AROME, "level" for MNH).

    Output
    ------
    A new dict with the same keys as dico_user, where only the height group dataset
    is replaced when collocation is applied:
      - MNH: replaces UT, VT, WT by uc, vc, wc
      - AROME: replaces w by wc
    If height group or required winds are missing, returns dico_user unchanged.

    Required conf attributes
    ------------------------
    - conf.atm_model
    - conf.name_time_dim
    - conf.name_vertical_dim
    - conf.name_lat_dim
    - conf.name_lon_dim
    - conf.velocity_aliases: {"u_velocity": ..., "v_velocity": ..., "w_velocity": ...}
    """
    if logger is None:
        logger = logging.getLogger(__name__)

    if boundary_policy not in {"partial", "strict"}:
        raise ValueError("boundary_policy must be 'partial' or 'strict'.")

    atm_model = str(getattr(conf, "atm_model")).upper()

    # Height-group key in dico_user is assumed to match the vertical dim name for height coordinates
    height_key = getattr(conf, "name_vertical_dim")

    if height_key not in dico_user:
        return dico_user
    ds_h = dico_user[height_key]

    vel_alias = getattr(conf, "velocity_aliases")
    u_name = vel_alias.get("u_velocity", "u")
    v_name = vel_alias.get("v_velocity", "v")
    w_name = vel_alias.get("w_velocity", "w")
    print(u_name,v_name,w_name)

    # Center dims as defined by the model configuration
    time_dim = getattr(conf, "name_time_dim", "time")
    yC = getattr(conf, "name_lat_dim")
    xC = getattr(conf, "name_lon_dim")
    zC = getattr(conf, "name_vertical_dim")

    print(yC,xC,zC)
    if atm_model == "AROME":
        # Only w -> wc in height group
        if w_name not in ds_h.data_vars:
            return dico_user

        wc0 = _arome_fastpath_wc(ds_h[w_name], z_center=zC, boundary_policy=boundary_policy)
        wc = _tag_collocation(wc0, boundary_policy).rename("wc")

        ds_h_out = ds_h.copy(deep=False).drop_vars([w_name], errors="ignore")
        ds_h_out["wc"] = wc

        _assert_center_dims(ds_h_out["wc"], xC=xC, yC=yC, zC=zC, name="wc")
        return _replace_height_group(dico_user, height_key, ds_h_out)

    if atm_model == "MNH":
        # Always xgcm, require u and v, w optional
        if u_name not in ds_h.data_vars or v_name not in ds_h.data_vars:
            return dico_user

        ds_h_out = _mnh_xgcm_collocation(
            ds_h,
            xC=xC,
            yC=yC,
            zC=zC,
            u_name=u_name,
            v_name=v_name,
            w_name=w_name,
            boundary_policy=boundary_policy,
        )
        return _replace_height_group(dico_user, height_key, ds_h_out)

    return dico_user


def _replace_height_group(
    dico_user: Mapping[str, xr.Dataset],
    height_key: str,
    ds_h_out: xr.Dataset,
) -> Mapping[str, xr.Dataset]:
    out: MutableMapping[str, xr.Dataset] = dict(dico_user)
    out[height_key] = ds_h_out
    return out


def _tag_collocation(da: xr.DataArray, boundary_policy: str) -> xr.DataArray:
    da = da.copy()
    da.attrs = dict(da.attrs)
    da.attrs["collocation"] = "xgcm interp to cell_center"
    da.attrs["vertical_collocation"] = "interfaces_to_centers"
    da.attrs["boundary_policy"] = boundary_policy
    return da


def _assert_center_dims(
    da: xr.DataArray,
    *,
    xC: str,
    yC: str,
    zC: str,
    name: str,
) -> None:
    if da.ndim >= 3:
        if da.dims[-2] != yC or da.dims[-1] != xC:
            raise ValueError(
                f"{name} is not on expected horizontal center dims, "
                f"got (...,{da.dims[-2]},{da.dims[-1]}) expected (...,{yC},{xC})."
            )
    if da.ndim >= 4:
        if da.dims[-3] != zC:
            raise ValueError(
                f"{name} is not on expected vertical center dim, got {da.dims[-3]} expected {zC}."
            )


def _arome_fastpath_wc(
    w: xr.DataArray,
    *,
    z_center: str,
    boundary_policy: str,
) -> xr.DataArray:
    """
    AROME: vertical collocation only.
    If w already uses z_center, return as-is.
    If w uses an interface dim with size Nz+1 relative to z_center, reduce by pairwise mean.
    """
    if w.ndim < 4:
        return w

    z_src = w.dims[-3]
    if z_src == z_center:
        return w

    # If the target dim does not exist in the DataArray, only allow rename
    if z_center not in w.sizes:
        return w.rename({z_src: z_center})

    n_src = w.sizes[z_src]
    n_tgt = w.sizes[z_center]

    if n_src == n_tgt:
        return w.rename({z_src: z_center})

    if n_src == n_tgt + 1:
        w0 = w.isel({z_src: slice(0, n_tgt)})
        w1 = w.isel({z_src: slice(1, n_tgt + 1)})
        wc = _pairwise_mean(w0, w1, policy=boundary_policy)
        return wc.rename({z_src: z_center})

    raise ValueError(
        f"AROME vertical collocation failed: {z_src!r} size {n_src} cannot be mapped to "
        f"{z_center!r} size {n_tgt}."
    )


def _pairwise_mean(a0: xr.DataArray, a1: xr.DataArray, *, policy: str) -> xr.DataArray:
    if policy == "strict":
        ok = np.isfinite(a0) & np.isfinite(a1)
        return xr.where(ok, 0.5 * (a0 + a1), np.nan)

    a0_ok = np.isfinite(a0)
    a1_ok = np.isfinite(a1)
    cnt = a0_ok.astype(np.int8) + a1_ok.astype(np.int8)
    s = xr.where(a0_ok, a0, 0.0) + xr.where(a1_ok, a1, 0.0)
    return xr.where(cnt > 0, s / cnt, np.nan)


def _mnh_xgcm_collocation(
    ds_h: xr.Dataset,
    *,
    xC: str,
    yC: str,
    zC: str,
    u_name: str,
    v_name: str,
    w_name: str,
    boundary_policy: str,
) -> xr.Dataset:
    """
    MNH: xgcm collocation to centers (yC, xC, zC).
    Assumes staggering is encoded by distinct dim names (e.g. ni_u, nj_u, level_w).
    """
    u = ds_h[u_name]
    v = ds_h[v_name]
    print(u)

    coords = {"X": {"center": xC}, "Y": {"center": yC}, "Z": {"center": zC}}

    # X staggering from u, typically ni_u
    if u.dims[-1] != xC:
        coords["X"]["left"] = u.dims[-1]

    # Y staggering from v, typically nj_v
    if v.dims[-2] != yC:
        coords["Y"]["left"] = v.dims[-2]

    # Z staggering from w, typically level_w
    if w_name in ds_h.data_vars and ds_h[w_name].ndim >= 4:
        w = ds_h[w_name]
        zW = w.dims[-3]
        if zW != zC:
            coords["Z"]["outer"] = zW

        # If w is also staggered horizontally, add left dims if not already set
        if w.dims[-1] != xC and "left" not in coords["X"]:
            coords["X"]["left"] = w.dims[-1]
        if w.dims[-2] != yC and "left" not in coords["Y"]:
            coords["Y"]["left"] = w.dims[-2]

    grid = Grid(ds_h, coords=coords, periodic=False, autoparse_metadata=False)

    def interp_to_center(da: xr.DataArray) -> xr.DataArray:
        out = da
        if out.dims[-1] != xC:
            out = _interp_nanaware(grid, out, axis="X", to="center", policy=boundary_policy)
        if out.dims[-2] != yC:
            out = _interp_nanaware(grid, out, axis="Y", to="center", policy=boundary_policy)
        if out.ndim >= 4 and out.dims[-3] != zC:
            out = _interp_nanaware(grid, out, axis="Z", to="center", policy=boundary_policy)
        return out

    uc = _tag_collocation(interp_to_center(u), boundary_policy).rename("uc")
    vc = _tag_collocation(interp_to_center(v), boundary_policy).rename("vc")

    ds_out = ds_h.copy(deep=False).drop_vars([u_name, v_name], errors="ignore")
    ds_out["uc"] = uc
    ds_out["vc"] = vc

    _assert_center_dims(ds_out["uc"], xC=xC, yC=yC, zC=zC, name="uc")
    _assert_center_dims(ds_out["vc"], xC=xC, yC=yC, zC=zC, name="vc")

    if w_name in ds_h.data_vars:
        wc = _tag_collocation(interp_to_center(ds_h[w_name]), boundary_policy).rename("wc")
        ds_out = ds_out.drop_vars([w_name], errors="ignore")
        ds_out["wc"] = wc
        _assert_center_dims(ds_out["wc"], xC=xC, yC=yC, zC=zC, name="wc")

    return ds_out


def _interp_nanaware(
    grid: Grid,
    da: xr.DataArray,
    *,
    axis: str,
    to: str,
    policy: str,
) -> xr.DataArray:
    """
    xgcm interpolation with NaN policy.
    strict: NaNs propagate, boundary fill is NaN.
    partial: weighted interpolation ignoring NaNs, boundary fill is 0 for data and weights.
    """
    if policy == "strict":
        return grid.interp(da, axis=axis, to=to, boundary="fill", fill_value=np.nan)

    finite = np.isfinite(da)
    w = xr.where(finite, 1.0, 0.0)
    da0 = xr.where(finite, da, 0.0)

    da_i = grid.interp(da0, axis=axis, to=to, boundary="fill", fill_value=0.0)
    w_i = grid.interp(w, axis=axis, to=to, boundary="fill", fill_value=0.0)
    return xr.where(w_i > 0, da_i / w_i, np.nan)


In [ ]:


dico_user = runner.ds_box

dico_for_polar = collocate_winds(dico_user, conf=conf)